# Chapitre 2 — Construire un premier RAG

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ahouahounko/rag-en-pratique/blob/main/chapters/chapitre-02-premier-rag/02_premier_rag.ipynb)

Ce notebook met en œuvre ingestion, chunking, embeddings, retrieval, génération et citations. Le mode hors ligne est actif par défaut.

## 1. Préparer le dépôt

La cellule fonctionne dans Colab et depuis la racine du dépôt.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

if not Path("src").is_dir():
    if not Path("rag-en-pratique").is_dir():
        subprocess.run(["git", "clone", "https://github.com/Ahouahounko/rag-en-pratique.git"], check=True)
    os.chdir("rag-en-pratique")

sys.path.insert(0, str(Path("src").resolve()))
print("Dépôt prêt :", Path.cwd())


## 2. Charger les documents de démonstration

In [ ]:
from pathlib import Path
from rag_en_pratique.core import Document

data_directory = Path("data/sample")
documents = [
    Document(path.read_text(encoding="utf-8"), {"source": path.name})
    for path in sorted(data_directory.glob("*.md"))
    if path.name.lower() != "readme.md"
]
[(doc.metadata["source"], len(doc.text)) for doc in documents]


## 3. Découper les documents

In [ ]:
from rag_en_pratique.core import split_documents

chunks = split_documents(documents, chunk_size=60, overlap=10)
print(f"{len(documents)} documents -> {len(chunks)} chunks")
chunks[0]


## 4. Indexer et rechercher hors ligne

In [ ]:
from rag_en_pratique.core import HashingEmbedder, InMemoryVectorStore

store = InMemoryVectorStore(HashingEmbedder(dimensions=256))
store.add(chunks)
results = store.search("Quel est le délai pour retourner un produit ?", top_k=3)
[(round(item.score, 3), item.document.metadata["source"]) for item in results]


## 5. Assembler le pipeline RAG

In [ ]:
from rag_en_pratique.core import ExtractiveGenerator, RAGPipeline

rag = RAGPipeline(store, ExtractiveGenerator())
response = rag.ask("Sous combien de jours peut-on retourner un produit ?")
print(response["answer"])


## 6. Examiner les sources

Une application RAG doit rendre ses sources inspectables.

In [ ]:
for source in response["sources"]:
    print(source["score"], source["metadata"]["source"])
    print(source["text"][:250])
    print()


## 7. Activer OpenAI plus tard (facultatif)

La cellule ne fait aucun appel tant que `USE_OPENAI` vaut `False`. Configurez `OPENAI_API_KEY` et `OPENAI_MODEL` dans votre environnement, sans les inscrire dans le notebook.

In [ ]:
USE_OPENAI = False

if USE_OPENAI:
    from rag_en_pratique.openai_adapter import OpenAIEmbedder, OpenAIGenerator
    from rag_en_pratique.core import InMemoryVectorStore, RAGPipeline

    online_store = InMemoryVectorStore(OpenAIEmbedder())
    online_store.add(chunks)
    online_rag = RAGPipeline(online_store, OpenAIGenerator())
    online_response = online_rag.ask("Sous combien de jours peut-on retourner un produit ?")
    print(online_response["answer"])
else:
    print("Mode OpenAI désactivé : aucun appel API effectué.")


## Pour aller plus loin

Comparez plusieurs tailles de chunks, modifiez `top_k`, ajoutez un document et vérifiez que la réponse cite toujours la bonne source.